In [24]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [25]:
load_dotenv()


model = ChatOpenAI(model = 'gpt-5-nano')

In [26]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    evaluate: str

In [28]:
def create_outline(state: BlogState) -> BlogState:
    #fetch title
    title =  state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # updata state
    state['outline'] = outline

    return state

In [31]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog under 3000 words on the title - {title} using the following \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [32]:
def evaluate(state: BlogState) -> BlogState:
    outline = state['outline']
    content = state['content'].split()
    content = ' '.join(content[:round(len(content)/2)])

    prompt = f'Based on this outline {outline} rate my blog out of 10 \n blog - {content} '

    eval = model.invoke(prompt).content
    state['evaluate'] = eval

    return state

In [33]:
graph =  StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)

graph.add_node('create_blog', create_blog)

graph.add_node('evaluate', evaluate)


# edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate', END)

workflow = graph.compile()

In [34]:
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{
    'title': 'Rise of AI in India',
    'outline': 'Here is a detailed, ready-to-use outline for a blog post or a multi-part series on the topic "Rise 
of AI in India." It covers context, sectors, policy, industry, ethics, challenges, and practical takes for various 
readers. Use it as a skeleton and fill in with current data, case studies, and local insights.\n\nTitle ideas (pick
one or mix and match)\n- The Rise of AI in India: Tech Leap, Policy Push, and Economic Impact\n- India and the AI 
Era: From Grassroots to Global Stage\n- AI in India: Transformation Across Sectors and the Path Forward\n- Building
an AI-Powered India: Opportunities, Challenges, and the Road Ahead\n\nCore thesis (one-sentence to guide the 
piece)\n- India is rapidly adopting and shaping AI through a unique mix of startup innovation, enterprise-led 
deployments, and government policy, with significant implications for jobs, governance, and inclusive 
growth.\n\nDetailed outline\n\n1) Introduction\n- Hook: The moment India stands at the intersection of vast digital
infrastructure, a large talent pool, and government-led AI ambition.\n- Why now: Advances in data generation, 
mobile adoption, cloud computing, and an action-ready policy environment.\n- What readers will gain: A clear view 
of where AI is taking root in India, which sectors are most affected, and what this means for businesses, 
policymakers, and individuals.\n- Scope and caveats: Acknowledge local challenges (digital divide, data governance,
infrastructure) and the need for responsible AI.\n\n2) The current AI landscape in India\n- Drivers of AI 
adoption\n  - Digital penetration: smartphone ownership, internet access, and data availability.\n  - Enterprise 
readiness: IT services firms, product companies increasingly embedding AI in offerings.\n  - Availability of data 
and talent: universities, research labs, and large engineering workforce.\n- Policy and regulatory environment 
(high-level)\n  - Government vision and strategic direction for AI.\n  - Focus areas such as data governance, 
privacy, ethics, and public-sector AI applications.\n- Investment and ecosystem\n  - Growing venture funding, 
corporate R&D, and AI accelerators/incubators.\n  - Role of academic institutions and think tanks in AI research 
and skilling.\n- Workforce implications\n  - Demand for data scientists, AI engineers, ML ops, and domain AI 
specialists.\n  - Emphasis on reskilling and continuous learning.\n\n3) Sectors where AI is making an impact 
(structure by sector; provide 2–4 concrete examples per sector)\n- Healthcare\n  - Applications: diagnostic 
support, imaging analysis, remote monitoring, drug discovery aids.\n  - Potential impact: faster diagnosis, 
improved access in underserved areas, cost reductions.\n  - Challenges: data privacy, regulatory approvals, 
interoperability of health records.\n- Agriculture\n  - Applications: crop health monitoring, yield optimization, 
precision agriculture, supply-chain traceability.\n  - Benefits: higher yields, reduced input waste, farmer 
empowerment.\n  - Challenges: data collection in rural areas, equipment costs, weather variability.\n- Financial 
services and fintech\n  - Applications: credit scoring from alternative data, fraud detection, customer service 
chatbots, personalized financial advice.\n  - Benefits: financial inclusion, faster underwriting, better risk 
management.\n  - Challenges: data privacy, bias in models, regulatory compliance.\n- Manufacturing and logistics\n 
- Applications: predictive maintenance, quality control, demand forecasting, route optimization.\n  - Benefits: 
lower downtime, efficiency gains, smarter warehouses.\n  - Challenges: capital costs, integration with legacy 
systems.\n- Education and skilling\n  - Applications: adaptive learning platforms, automated assessment, skill 
mapping and upskilling tracks.\n  - Benefits: personalized learning, scalable upskilling.\n  - Challenges: digital 
divide, pedagogy alignment, teach

In [35]:
print(final_state['evaluate'])

Rate: 8/10

What’s strong
- Ambitious, well-structured scope: It covers the big picture (policy, ecosystem, sectors, ethics, workforce, 
future), not just hype.
- Sector depth: Provides concrete AI use cases across healthcare, agriculture, fintech, manufacturing, education, 
public sector, and retail—useful for diverse readers.
- Practical framing: Includes readers’ takeaways, skills paths, and governance/ethics angle, which helps readers 
translate ideas into action.
- Local context cues: Mentions Indian hubs (Bengaluru, Pune, Hyderabad, Chennai, Mumbai, NCR) and the broad 
ecosystem, which anchors AI to India’s geography.
- Ready-to-use skeleton: The outline is actionable and easy to map into sections, graphics, case studies, and pull 
quotes.

Key gaps and opportunities for improvement
- Current-data gaps: The draft uses general statements but would be stronger with up-to-date data points (market 
size, investment volumes, talent pipelines, regulatory milestones, pilot programs) and citations from credible 
sources (NITI Aayog, DPDP updates, NASSCOM–PwC, industry reports, academic centers).
- Cohesion and flow: Some sections read like a list of bullets. A stronger narrative thread (e.g., “AI as an 
enabler of inclusive growth across sectors, with a timeline of policy, pilots, and market ads”) will improve 
readability.
- Completeness in places: The Agriculture AI subsection in your draft is cut off (“Intello Labs: computer 
vision-based quality assessment for produce and crops, enabling better...”). It needs a complete sentence and a few
more examples to avoid a sense of unfinished content.
- Local evidence and variety: More state/region examples (e.g., Karnataka, Maharashtra, Tamil Nadu, Andhra Pradesh,
Telangana) and tier-2/3 city initiatives would enhance relevance for readers outside the metro hubs.
- Voices and sources: Integrate quotes or mini case-study snapshots from policymakers, industry leaders, and 
researchers with citations to keep authority high.
- Data visualization plan: You mention visuals; explicitly map out what visuals will go where (timelines, sector 
impact charts, regional maps, micro-case cards) and align with the data you’ll cite.
- Tonal balance: Strong headline and policy framing can drift toward optimism. Include more explicit counterpoints 
or cautionary notes where appropriate (infrastructure gaps, data governance challenges, inclusion gaps).

Concrete, actionable edits you can apply now
- Finish and strengthen every bullet under “The current AI landscape in India,” especially with current numbers 
(smartphone penetration, internet user base, data growth, AI-related job postings, venture funding in AI, number of
AI startups, notable exits or scale stories). Add citations.
- Complete the Agriculture AI line: finish the sentence and add a couple more examples (e.g., CropIn, Intello Labs,
e-Krishi ecosystem, farmers’ co-ops pilots) with a one-liner on impact and scale.
- Add a short “Why now” data point in the introduction with a recent statistic (e.g., number of AI-enabled pilot 
programs in public services in the last 18–24 months, government funding rounds, or a notable collaboration).
- Include at least two micro-cases per sector with numbers (e.g., time-to-diagnosis, yield uplift, cost savings, 
ROI) and a one-sentence takeaway for each.
- Insert a short “data privacy and governance” box in the policy section with real-world considerations India is 
currently debating (localization, cross-border data flows, and interoperability standards).
- Add a regional map showing AI activity by state or city, highlighting Bengaluru, Pune, Hyderabad, Chennai, and 
NCR as hubs, plus a sidebar noting notable tier-2/3 initiatives.
- Conclude with a concise, action-oriented takeaway for each audience segment (students, startups, corporates, 
policymakers, citizens) as you already propose, but tailor each takeaway with one concrete next step and a quick 
resource list.

A quick, filled-in snippet to help visu

In [36]:
print(final_state['content'])

The Rise of AI in India: Tech Leap, Policy Push, and Economic Impact

Introduction
India stands at a defining crossroads: a vast digital infrastructure, a young, tech-savvy population, and a 
government that has placed AI high on the national agenda. It’s a moment where global innovations meet local 
realities—where a billion plus people create new data streams, and where policy, startups, and large enterprises 
are learning to harness AI for inclusive growth. The promise is not just faster apps or smarter chatbots; it’s the 
potential to improve healthcare, farming, education, financial inclusion, and public services at scale.

Why now? Several forces are converging. Widespread mobile adoption, expanding internet access, and the rollout of 
affordable cloud services have lowered the barriers to AI experimentation. A growing ecosystem of AI startups, 
product companies embedding AI in offerings, and a robust services sector increasingly delivering AI-enabled 
solutions create a powerful feedback loop. On the policy side, India has outlined a strategic vision for AI, 
emphasizing governance, ethics, data, and public-sector applications, while continuing to participate in global 
conversations about responsible AI.

What you’ll gain from this piece: a clear map of where AI is taking root in India, which sectors are most affected,
and what this means for businesses, policymakers, and individuals. We’ll ground the discussion in real-world 
examples, current initiatives, and practical implications for readers across categories—students, professionals, 
startups, large enterprises, and citizens.

The current AI landscape in India

Drivers of AI adoption
- Digital penetration and data generation: India’s enormous and growing digital audience—hundreds of millions 
online across mobile networks—provides a rich data substrate for AI. This data fuels smarter apps, personalized 
services, and more accurate models across sectors.
- Enterprise readiness: India’s IT services ecosystem, product engineering firms, and AI-first startups are 
increasingly embedding AI into offerings, from business processes to domain-specific solutions.
- Availability of data and talent: universities, dedicated research labs, and a deep pool of software engineers 
create a favorable talent supply for AI development. There’s rising demand for data scientists, ML engineers, ML 
operations (MLOps), and domain AI specialists.

Policy and regulatory environment (high-level)
- Vision and strategy: India frames AI as an enabler of inclusive growth, innovation, and national competitiveness.
The intent is to accelerate AI R&D, foster public-private partnerships, and drive public service improvements while
maintaining strong governance.
- Focus areas: privacy and data governance, ethics, interoperability, and public-sector AI applications are central
to policy discussions. The aim is to balance innovation with protections for citizens and data sovereignty.
- Data governance scaffolding: efforts to standardize data formats and enable secure cross-sector data use, while 
guarding privacy and security, are part of the policy conversation.
- Talent and research: public funding for AI education, research centers, and industry-academia collaboration 
increases the supply of skilled AI professionals and researchers.

Investment and ecosystem
- Funding and experimentation: a growing stream of venture funding, corporate R&D investment, and 
accelerators/incubators supports AI startups and pilot projects in large organizations.
- Academic and think-tank role: universities, AI labs, and think tanks contribute to foundational research, applied
projects, and skilling initiatives that align with market needs.
- Notable hubs: major AI activity centers in Bengaluru, Pune, Hyderabad, Chennai, Mumbai, and NCR regions, with 
growing presence in tier-2/3 cities through state-led programs and private investment.

Workforce implications
- Demand for talent: strong demand for data scientists, M